# BirdCLEF 2026 — Audio Classification

**Competition:** [BirdCLEF 2026](https://www.kaggle.com/competitions/birdclef-2026)  
**Task:** Multi-label species classification from soundscape recordings  
**Ecosystem:** Pantanal wetlands, South America  
**Species:** 234 (birds, amphibians, mammals, reptiles, insects)  
**Evaluation Metric:** Macro ROC-AUC

---

## Pipeline Overview

```
Raw Audio (.ogg)  →  Mel Spectrogram  →  EfficientNet-B2  →  Multi-label Predictions
```

**Key techniques:**
- Mel spectrogram extraction with `librosa`
- EfficientNet-B2 backbone via `timm` (ImageNet pretrained)
- GeM pooling classification head
- Audio augmentations: time shift, Gaussian noise, mixup
- BCEWithLogits loss (multi-label)
- Mixed precision training (AMP)
- Cosine annealing with warmup
- StratifiedKFold cross-validation
- Test-Time Augmentation (TTA)


---
## 1. Setup & Imports

In [ ]:
# Install / upgrade key dependencies if running on Kaggle kernel
import subprocess, sys

def pip_install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

# Uncomment on first run in a fresh Kaggle environment
# pip_install("timm>=0.9.12")
# pip_install("audiomentations")
# pip_install("librosa")

In [ ]:
# ── Standard library ──────────────────────────────────────────────────────────
import os
import gc
import math
import random
import warnings
import time
from pathlib import Path
from collections import defaultdict

warnings.filterwarnings("ignore")

# ── Numeric / data ────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd

# ── Visualisation ─────────────────────────────────────────────────────────────
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

matplotlib.rcParams["figure.dpi"] = 120
sns.set_theme(style="darkgrid", palette="muted")

# ── Audio ─────────────────────────────────────────────────────────────────────
import librosa
import librosa.display
import torchaudio
import torchaudio.transforms as AT

# ── ML / sklearn ──────────────────────────────────────────────────────────────
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, f1_score, confusion_matrix
from sklearn.preprocessing import LabelEncoder

# ── PyTorch ───────────────────────────────────────────────────────────────────
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast
import torchvision.transforms as T

# ── timm ──────────────────────────────────────────────────────────────────────
import timm

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42

def seed_everything(seed: int = SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything()

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    print(f"  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"  PyTorch:  {torch.__version__}")
print(f"  librosa:  {librosa.__version__}")
print(f"  timm:     {timm.__version__}")

---
## 2. Configuration

All hyperparameters and path definitions live in a single `Config` dataclass so they are easy to tune and track with MLflow.

In [ ]:
from dataclasses import dataclass, field
from typing import List, Optional, Tuple


@dataclass
class Config:
    # ── Paths ─────────────────────────────────────────────────────────────────
    # Adjust BASE_DIR to point at your Kaggle input or local data folder.
    BASE_DIR: Path = Path("/kaggle/input/birdclef-2026")
    TRAIN_AUDIO_DIR: Path = BASE_DIR / "train_audio"
    TEST_DIR: Path = BASE_DIR / "test_soundscapes"
    TRAIN_META_CSV: Path = BASE_DIR / "train_metadata.csv"
    SAMPLE_SUB_CSV: Path = BASE_DIR / "sample_submission.csv"
    OUTPUT_DIR: Path = Path("/kaggle/working")

    # ── Audio ─────────────────────────────────────────────────────────────────
    SR: int = 32_000          # Target sample rate (Hz)
    DURATION: float = 5.0     # Clip duration (seconds)
    N_SAMPLES: int = field(init=False)   # SR * DURATION

    # ── Mel spectrogram ───────────────────────────────────────────────────────
    N_MELS: int = 128
    N_FFT: int = 2048
    HOP_LENGTH: int = 512
    FMIN: float = 20.0
    FMAX: float = 16_000.0
    POWER: float = 2.0        # Power spectrogram
    TOP_DB: float = 80.0      # Dynamic range for dB conversion

    # ── Image dimensions fed to the CNN ───────────────────────────────────────
    IMG_HEIGHT: int = 128
    IMG_WIDTH: int = 312      # ≈ (SR * DURATION) / HOP_LENGTH

    # ── Model ─────────────────────────────────────────────────────────────────
    MODEL_NAME: str = "efficientnet_b2"
    PRETRAINED: bool = True
    NUM_CLASSES: int = 234     # 234 species in BirdCLEF 2026
    DROP_RATE: float = 0.3
    GEM_P: float = 3.0         # GeM pooling exponent

    # ── Training ──────────────────────────────────────────────────────────────
    EPOCHS: int = 30
    BATCH_SIZE: int = 32
    VAL_BATCH_SIZE: int = 64
    NUM_WORKERS: int = 4
    PIN_MEMORY: bool = True

    # ── Optimiser ─────────────────────────────────────────────────────────────
    LR: float = 1e-3
    LR_MIN: float = 1e-6
    WEIGHT_DECAY: float = 1e-4
    GRAD_CLIP: float = 1.0
    WARMUP_EPOCHS: int = 3

    # ── Loss ──────────────────────────────────────────────────────────────────
    LABEL_SMOOTHING: float = 0.05
    BCE_POS_WEIGHT: Optional[float] = None  # set after analysing class freq.

    # ── Augmentation ──────────────────────────────────────────────────────────
    AUGMENT_TRAIN: bool = True
    TIME_SHIFT_RATE: float = 0.4    # Fraction of clips to time-shift
    NOISE_RATE: float = 0.3         # Fraction of clips to add Gaussian noise
    MIXUP_ALPHA: float = 0.5        # Mixup beta distribution alpha
    FREQ_MASK_PARAM: int = 24       # SpecAugment frequency mask
    TIME_MASK_PARAM: int = 64       # SpecAugment time mask

    # ── Cross-validation ──────────────────────────────────────────────────────
    N_FOLDS: int = 5
    FOLD: int = 0              # Active fold for this run

    # ── Inference / TTA ───────────────────────────────────────────────────────
    TTA_STEPS: int = 5
    SCORE_THRESHOLD: float = 0.5

    # ── Early stopping ────────────────────────────────────────────────────────
    PATIENCE: int = 7
    MIN_DELTA: float = 1e-4

    def __post_init__(self):
        self.N_SAMPLES = int(self.SR * self.DURATION)
        self.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


CFG = Config()

print("Configuration summary:")
for k, v in CFG.__dict__.items():
    print(f"  {k:25s}: {v}")

---
## 3. Exploratory Data Analysis (EDA)

Examine the training metadata to understand class distribution, recording lengths, geographic spread, and a few raw audio samples.

In [ ]:
# Load training metadata
meta = pd.read_csv(CFG.TRAIN_META_CSV)
print(f"Shape: {meta.shape}")
meta.head()

In [ ]:
print("Columns:", meta.columns.tolist())
print("\nMissing values:")
print(meta.isnull().sum())
print(f"\nUnique species: {meta['primary_label'].nunique()}")
print(f"Total recordings: {len(meta)}")

In [ ]:
# ── Class distribution ────────────────────────────────────────────────────────
species_counts = meta["primary_label"].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(18, 5))

# Top-40 most common species
top40 = species_counts.head(40)
axes[0].barh(top40.index[::-1], top40.values[::-1], color="steelblue", edgecolor="white", linewidth=0.4)
axes[0].set_title("Top-40 Most Common Species", fontsize=13)
axes[0].set_xlabel("Number of recordings")
axes[0].tick_params(axis="y", labelsize=7)

# Distribution histogram
axes[1].hist(species_counts.values, bins=50, color="coral", edgecolor="white", linewidth=0.6)
axes[1].set_title("Recording Count Distribution per Species", fontsize=13)
axes[1].set_xlabel("Recordings per species")
axes[1].set_ylabel("Number of species")
axes[1].axvline(species_counts.mean(), color="navy", linestyle="--", label=f"Mean: {species_counts.mean():.0f}")
axes[1].axvline(species_counts.median(), color="darkred", linestyle="--", label=f"Median: {species_counts.median():.0f}")
axes[1].legend()

plt.suptitle("BirdCLEF 2026 — Class Distribution", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

print(f"Min recordings: {species_counts.min()} ({species_counts.idxmin()})")
print(f"Max recordings: {species_counts.max()} ({species_counts.idxmax()})")
print(f"Imbalance ratio (max/min): {species_counts.max() / species_counts.min():.1f}x")

In [ ]:
# ── Geographic distribution ───────────────────────────────────────────────────
if {"latitude", "longitude"}.issubset(meta.columns):
    fig, ax = plt.subplots(figsize=(10, 6))
    sc = ax.scatter(
        meta["longitude"].dropna(),
        meta["latitude"].dropna(),
        c=meta["latitude"].dropna(),
        cmap="viridis",
        alpha=0.35,
        s=6,
    )
    plt.colorbar(sc, ax=ax, label="Latitude")
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    ax.set_title("Recording Locations (Pantanal region)")
    plt.tight_layout()
    plt.show()

In [ ]:
# ── Sample waveforms & mel-spectrograms ───────────────────────────────────────
# Pick 4 random species and show one recording each
sample_species = meta["primary_label"].drop_duplicates().sample(4, random_state=SEED).tolist()

fig = plt.figure(figsize=(18, 12))
gs = gridspec.GridSpec(4, 2, figure=fig, hspace=0.55, wspace=0.3)

for row_idx, sp in enumerate(sample_species):
    row = meta[meta["primary_label"] == sp].iloc[0]
    audio_path = CFG.TRAIN_AUDIO_DIR / row["filename"]

    try:
        y, sr = librosa.load(str(audio_path), sr=CFG.SR, duration=CFG.DURATION, mono=True)
    except Exception as e:
        print(f"  Could not load {audio_path}: {e}")
        continue

    # Waveform
    ax_wave = fig.add_subplot(gs[row_idx, 0])
    times = np.linspace(0, len(y) / sr, num=len(y))
    ax_wave.plot(times, y, color="steelblue", linewidth=0.6)
    ax_wave.set_title(f"{sp} — waveform", fontsize=10)
    ax_wave.set_xlabel("Time (s)")
    ax_wave.set_ylabel("Amplitude")

    # Mel spectrogram
    ax_mel = fig.add_subplot(gs[row_idx, 1])
    mel = librosa.feature.melspectrogram(
        y=y, sr=sr,
        n_mels=CFG.N_MELS,
        n_fft=CFG.N_FFT,
        hop_length=CFG.HOP_LENGTH,
        fmin=CFG.FMIN,
        fmax=CFG.FMAX,
        power=CFG.POWER,
    )
    mel_db = librosa.power_to_db(mel, ref=np.max, top_db=CFG.TOP_DB)
    img = librosa.display.specshow(
        mel_db, sr=sr, hop_length=CFG.HOP_LENGTH,
        x_axis="time", y_axis="mel",
        fmin=CFG.FMIN, fmax=CFG.FMAX,
        ax=ax_mel, cmap="magma",
    )
    ax_mel.set_title(f"{sp} — mel-spectrogram", fontsize=10)
    fig.colorbar(img, ax=ax_mel, format="%+2.0f dB", pad=0.01)

fig.suptitle("Sample Recordings — Waveforms & Mel Spectrograms", fontsize=14, fontweight="bold")
plt.show()

---
## 4. Audio Preprocessing

Functions for:
1. Loading & resampling audio
2. Padding / truncating to a fixed number of samples
3. Mel-spectrogram conversion (→ 3-channel image for CNN)
4. Audio-domain augmentations (time shift, Gaussian noise, mixup)

In [ ]:
# ── 4.1  Audio loading utilities ──────────────────────────────────────────────

def load_audio(path: str, sr: int = CFG.SR, duration: float = CFG.DURATION) -> np.ndarray:
    """Load an audio file and return a mono float32 numpy array."""
    y, _ = librosa.load(path, sr=sr, duration=duration, mono=True, res_type="kaiser_fast")
    return y.astype(np.float32)


def pad_or_truncate(y: np.ndarray, target_len: int = CFG.N_SAMPLES) -> np.ndarray:
    """
    Ensure the waveform has exactly `target_len` samples.
    - Too short → repeat-pad (tile) rather than zero-pad so the network sees
      realistic energy across the full clip.
    - Too long → random crop from a uniformly sampled start position.
    """
    n = len(y)
    if n < target_len:
        repeats = math.ceil(target_len / n)
        y = np.tile(y, repeats)
    if len(y) > target_len:
        start = random.randint(0, len(y) - target_len)
        y = y[start: start + target_len]
    return y[:target_len]

In [ ]:
# ── 4.2  Mel-spectrogram conversion ───────────────────────────────────────────

def audio_to_melspec(y: np.ndarray, cfg: Config = CFG) -> np.ndarray:
    """
    Convert a waveform to a normalised mel-spectrogram in dB.
    Returns a float32 array of shape (n_mels, time_frames).
    """
    mel = librosa.feature.melspectrogram(
        y=y,
        sr=cfg.SR,
        n_mels=cfg.N_MELS,
        n_fft=cfg.N_FFT,
        hop_length=cfg.HOP_LENGTH,
        fmin=cfg.FMIN,
        fmax=cfg.FMAX,
        power=cfg.POWER,
    )
    mel_db = librosa.power_to_db(mel, ref=np.max, top_db=cfg.TOP_DB)
    # Normalise to [0, 1]
    mel_db = (mel_db + cfg.TOP_DB) / cfg.TOP_DB
    return mel_db.astype(np.float32)


def melspec_to_tensor(mel: np.ndarray, img_h: int = CFG.IMG_HEIGHT, img_w: int = CFG.IMG_WIDTH) -> torch.Tensor:
    """
    Resize the spectrogram to a fixed canvas and stack as a 3-channel tensor
    (mimicking RGB) so standard ImageNet-pretrained backbones can be used.
    Shape: (3, img_h, img_w)
    """
    # mel: (n_mels, time)
    tensor = torch.from_numpy(mel).unsqueeze(0)          # (1, n_mels, time)
    tensor = F.interpolate(
        tensor.unsqueeze(0),
        size=(img_h, img_w),
        mode="bilinear",
        align_corners=False,
    ).squeeze(0)                                          # (1, img_h, img_w)
    tensor = tensor.repeat(3, 1, 1)                       # (3, img_h, img_w)
    return tensor

In [ ]:
# ── 4.3  Audio augmentations ──────────────────────────────────────────────────

def time_shift(y: np.ndarray, max_shift: float = 0.5) -> np.ndarray:
    """
    Randomly roll the waveform by up to `max_shift` fraction of its length.
    Rolled samples wrap around, maintaining the same clip duration.
    """
    shift = int(random.uniform(-max_shift, max_shift) * len(y))
    return np.roll(y, shift)


def add_gaussian_noise(y: np.ndarray, snr_db_range: Tuple[float, float] = (20.0, 40.0)) -> np.ndarray:
    """
    Add white Gaussian noise at a uniformly sampled SNR (in dB).
    """
    snr_db = random.uniform(*snr_db_range)
    signal_power = np.mean(y ** 2) + 1e-9
    noise_power = signal_power / (10 ** (snr_db / 10))
    noise = np.random.randn(*y.shape).astype(np.float32) * math.sqrt(noise_power)
    return np.clip(y + noise, -1.0, 1.0)


def mixup_audio(
    y1: np.ndarray, y2: np.ndarray,
    label1: np.ndarray, label2: np.ndarray,
    alpha: float = CFG.MIXUP_ALPHA,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Mixup in the waveform domain.
    Returns (mixed_audio, mixed_labels).
    """
    lam = np.random.beta(alpha, alpha)
    y_mix = lam * y1 + (1.0 - lam) * y2
    l_mix = lam * label1 + (1.0 - lam) * label2
    return y_mix.astype(np.float32), l_mix.astype(np.float32)


def apply_spec_augment(
    spec: torch.Tensor,
    freq_mask_param: int = CFG.FREQ_MASK_PARAM,
    time_mask_param: int = CFG.TIME_MASK_PARAM,
    num_freq_masks: int = 2,
    num_time_masks: int = 2,
) -> torch.Tensor:
    """
    SpecAugment: random frequency & time masking on a spectrogram tensor.
    Input shape: (C, H, W).
    """
    _, h, w = spec.shape

    for _ in range(num_freq_masks):
        f = random.randint(0, freq_mask_param)
        f0 = random.randint(0, max(0, h - f))
        spec[:, f0: f0 + f, :] = 0.0

    for _ in range(num_time_masks):
        t = random.randint(0, time_mask_param)
        t0 = random.randint(0, max(0, w - t))
        spec[:, :, t0: t0 + t] = 0.0

    return spec

In [ ]:
# ── Quick sanity check: visualise augmentations ───────────────────────────────
sample_row = meta.iloc[0]
sample_path = str(CFG.TRAIN_AUDIO_DIR / sample_row["filename"])

y_raw = load_audio(sample_path)
y_fixed = pad_or_truncate(y_raw)

augmentations = {
    "Original": y_fixed,
    "Time shift": time_shift(y_fixed),
    "+ Gaussian noise": add_gaussian_noise(y_fixed),
}

fig, axes = plt.subplots(len(augmentations), 2, figsize=(16, 9))

for i, (label, y_aug) in enumerate(augmentations.items()):
    mel = audio_to_melspec(y_aug)
    t = np.linspace(0, CFG.DURATION, len(y_aug))

    axes[i, 0].plot(t, y_aug, lw=0.5, color="steelblue")
    axes[i, 0].set_title(f"{label} — waveform", fontsize=9)
    axes[i, 0].set_xlabel("Time (s)")

    librosa.display.specshow(
        mel * CFG.TOP_DB - CFG.TOP_DB,
        sr=CFG.SR, hop_length=CFG.HOP_LENGTH,
        x_axis="time", y_axis="mel",
        fmin=CFG.FMIN, fmax=CFG.FMAX,
        ax=axes[i, 1], cmap="magma",
    )
    axes[i, 1].set_title(f"{label} — mel-spectrogram", fontsize=9)

plt.tight_layout()
plt.show()

---
## 5. Dataset & DataLoader

A custom `BirdDataset` handles:
- On-the-fly audio loading
- Mel-spectrogram conversion
- Optional waveform- and spectrogram-domain augmentations
- Mixup (when `use_mixup=True`)

Labels are stored as multi-hot float vectors so we can use BCEWithLogitsLoss for multi-label classification.

In [ ]:
# ── 5.1  Label encoding ───────────────────────────────────────────────────────
# Build a sorted list of all species (primary labels)
all_species = sorted(meta["primary_label"].unique().tolist())
species_to_idx = {sp: i for i, sp in enumerate(all_species)}
idx_to_species = {i: sp for sp, i in species_to_idx.items()}

NUM_CLASSES = len(all_species)
print(f"Number of classes: {NUM_CLASSES}")


def build_label_vector(primary_label: str, secondary_labels: str = "") -> np.ndarray:
    """
    Build a multi-hot label vector of shape (NUM_CLASSES,).
    primary_label is always 1.0; secondary_labels (space-separated) are 0.5
    to reflect weaker annotation confidence.
    """
    vec = np.zeros(NUM_CLASSES, dtype=np.float32)
    if primary_label in species_to_idx:
        vec[species_to_idx[primary_label]] = 1.0
    if isinstance(secondary_labels, str):
        for sp in secondary_labels.strip().split():
            if sp in species_to_idx:
                vec[species_to_idx[sp]] = 0.5
    return vec

In [ ]:
# ── 5.2  PyTorch Dataset ──────────────────────────────────────────────────────

class BirdDataset(Dataset):
    """
    Dataset for BirdCLEF 2026 training/validation.

    Parameters
    ----------
    df : pd.DataFrame
        Slice of train_metadata.csv with rows for this split.
    cfg : Config
        Global configuration object.
    mode : str
        'train' | 'val' | 'test'
    """

    def __init__(self, df: pd.DataFrame, cfg: Config = CFG, mode: str = "train"):
        self.df = df.reset_index(drop=True)
        self.cfg = cfg
        self.mode = mode
        self.is_train = (mode == "train")

        # Precompute label vectors to avoid repeated string parsing
        secondary_col = "secondary_labels" if "secondary_labels" in df.columns else None
        self.labels = np.stack([
            build_label_vector(
                row["primary_label"],
                row[secondary_col] if secondary_col else "",
            )
            for _, row in df.iterrows()
        ])  # (N, NUM_CLASSES)

    def __len__(self) -> int:
        return len(self.df)

    def _load_and_process(self, idx: int):
        row = self.df.iloc[idx]
        path = str(self.cfg.TRAIN_AUDIO_DIR / row["filename"])

        y = load_audio(path, sr=self.cfg.SR, duration=self.cfg.DURATION)
        y = pad_or_truncate(y, self.cfg.N_SAMPLES)

        # Waveform augmentations (train only)
        if self.is_train and self.cfg.AUGMENT_TRAIN:
            if random.random() < self.cfg.TIME_SHIFT_RATE:
                y = time_shift(y)
            if random.random() < self.cfg.NOISE_RATE:
                y = add_gaussian_noise(y)

        return y

    def __getitem__(self, idx: int):
        y = self._load_and_process(idx)
        label = self.labels[idx].copy()

        # Mixup (train only)
        if self.is_train and self.cfg.AUGMENT_TRAIN and random.random() < 0.3:
            mix_idx = random.randint(0, len(self) - 1)
            y2 = self._load_and_process(mix_idx)
            label2 = self.labels[mix_idx].copy()
            y, label = mixup_audio(y, y2, label, label2, alpha=self.cfg.MIXUP_ALPHA)

        # Mel spectrogram → tensor
        mel = audio_to_melspec(y, self.cfg)
        spec = melspec_to_tensor(mel, self.cfg.IMG_HEIGHT, self.cfg.IMG_WIDTH)

        # SpecAugment (train only)
        if self.is_train and self.cfg.AUGMENT_TRAIN:
            spec = apply_spec_augment(
                spec,
                freq_mask_param=self.cfg.FREQ_MASK_PARAM,
                time_mask_param=self.cfg.TIME_MASK_PARAM,
            )

        # ImageNet normalisation
        mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
        std  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
        spec = (spec - mean) / std

        return spec, torch.from_numpy(label)


class BirdTestDataset(Dataset):
    """Minimal dataset for test soundscapes (no labels)."""

    def __init__(self, file_list: List[str], cfg: Config = CFG):
        self.files = file_list
        self.cfg = cfg

    def __len__(self) -> int:
        return len(self.files)

    def __getitem__(self, idx: int):
        path = self.files[idx]
        y = load_audio(path, sr=self.cfg.SR, duration=self.cfg.DURATION)
        y = pad_or_truncate(y, self.cfg.N_SAMPLES)
        mel = audio_to_melspec(y, self.cfg)
        spec = melspec_to_tensor(mel, self.cfg.IMG_HEIGHT, self.cfg.IMG_WIDTH)
        mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
        std  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
        spec = (spec - mean) / std
        return spec, Path(path).stem

In [ ]:
# ── 5.3  Stratified K-Fold split ──────────────────────────────────────────────
skf = StratifiedKFold(n_splits=CFG.N_FOLDS, shuffle=True, random_state=SEED)

meta["fold"] = -1
for fold_idx, (_, val_idx) in enumerate(skf.split(meta, meta["primary_label"])):
    meta.loc[meta.index[val_idx], "fold"] = fold_idx

print("Fold distribution:")
print(meta.groupby("fold")["primary_label"].count())

train_df = meta[meta["fold"] != CFG.FOLD].reset_index(drop=True)
val_df   = meta[meta["fold"] == CFG.FOLD].reset_index(drop=True)
print(f"\nFold {CFG.FOLD}: train={len(train_df)}, val={len(val_df)}")

In [ ]:
# ── 5.4  DataLoaders ──────────────────────────────────────────────────────────
train_dataset = BirdDataset(train_df, cfg=CFG, mode="train")
val_dataset   = BirdDataset(val_df,   cfg=CFG, mode="val")

train_loader = DataLoader(
    train_dataset,
    batch_size=CFG.BATCH_SIZE,
    shuffle=True,
    num_workers=CFG.NUM_WORKERS,
    pin_memory=CFG.PIN_MEMORY,
    drop_last=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=CFG.VAL_BATCH_SIZE,
    shuffle=False,
    num_workers=CFG.NUM_WORKERS,
    pin_memory=CFG.PIN_MEMORY,
)

print(f"Train batches: {len(train_loader)}")
print(f"Val   batches: {len(val_loader)}")

# Quick shape check
specs, labels = next(iter(train_loader))
print(f"Batch spec shape:  {specs.shape}")
print(f"Batch label shape: {labels.shape}")

---
## 6. Model Architecture

EfficientNet-B2 is loaded from `timm` with ImageNet weights. The classifier head is replaced with:
1. **GeM Pooling** — Generalised Mean Pooling, which is generally superior to average pooling for fine-grained recognition tasks.
2. **Dropout** → **Linear(num_classes)** — multi-label output logits.

In [ ]:
# ── 6.1  GeM Pooling ──────────────────────────────────────────────────────────

class GeM(nn.Module):
    """
    Generalised Mean Pooling.
    Lernable parameter `p` controls the interpolation between
    average pooling (p=1) and max pooling (p→∞).
    """

    def __init__(self, p: float = 3.0, eps: float = 1e-6):
        super().__init__()
        self.p = nn.Parameter(torch.ones(1) * p)
        self.eps = eps

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, C, H, W)
        return F.avg_pool2d(
            x.clamp(min=self.eps).pow(self.p),
            kernel_size=(x.shape[-2], x.shape[-1]),
        ).pow(1.0 / self.p)

    def __repr__(self) -> str:
        return f"GeM(p={self.p.data.item():.4f}, eps={self.eps})"

In [ ]:
# ── 6.2  BirdClassifier ───────────────────────────────────────────────────────

class BirdClassifier(nn.Module):
    """
    EfficientNet-B2 backbone with GeM pooling and a custom multi-label head.
    """

    def __init__(self, cfg: Config = CFG):
        super().__init__()
        self.cfg = cfg

        # Load backbone (no global pool / classifier so we can insert GeM)
        self.backbone = timm.create_model(
            cfg.MODEL_NAME,
            pretrained=cfg.PRETRAINED,
            num_classes=0,          # remove classifier
            global_pool="",         # remove default pooling
            in_chans=3,
        )
        in_features = self.backbone.num_features

        self.pool = GeM(p=cfg.GEM_P)
        self.flatten = nn.Flatten()
        self.dropout = nn.Dropout(p=cfg.DROP_RATE)
        self.head = nn.Linear(in_features, cfg.NUM_CLASSES)

        # Initialise head with Xavier uniform
        nn.init.xavier_uniform_(self.head.weight)
        nn.init.zeros_(self.head.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, 3, H, W)
        features = self.backbone(x)   # (B, C, h, w)
        pooled   = self.pool(features) # (B, C, 1, 1)
        flat     = self.flatten(pooled)  # (B, C)
        dropped  = self.dropout(flat)
        logits   = self.head(dropped)    # (B, NUM_CLASSES)
        return logits


model = BirdClassifier(CFG).to(DEVICE)

# Parameter count
total  = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters:     {total:,}")
print(f"Trainable parameters: {trainable:,}")

# Forward pass sanity check
dummy = torch.randn(2, 3, CFG.IMG_HEIGHT, CFG.IMG_WIDTH).to(DEVICE)
with torch.no_grad():
    out = model(dummy)
print(f"Output shape: {out.shape}  (expected: [2, {CFG.NUM_CLASSES}])")

---
## 7. Training Loop

Training uses:
- **BCEWithLogitsLoss** with label smoothing
- **AdamW** optimiser
- **Linear warmup** followed by **CosineAnnealing**
- **Automatic Mixed Precision** (AMP) via `torch.cuda.amp`
- **Gradient clipping** to prevent gradient explosion
- **Early stopping** based on validation ROC-AUC

In [ ]:
# ── 7.1  Loss function ────────────────────────────────────────────────────────

class BCEWithLabelSmoothing(nn.Module):
    """
    BCE with label smoothing for multi-label classification.
    Smoothing shifts positive labels to (1 - eps) and negative to eps.
    """

    def __init__(self, smoothing: float = CFG.LABEL_SMOOTHING, reduction: str = "mean"):
        super().__init__()
        self.smoothing = smoothing
        self.reduction = reduction

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        targets_smooth = targets * (1.0 - self.smoothing) + 0.5 * self.smoothing
        loss = F.binary_cross_entropy_with_logits(
            logits, targets_smooth, reduction=self.reduction
        )
        return loss

In [ ]:
# ── 7.2  Learning-rate scheduler with warmup ──────────────────────────────────

def build_scheduler(optimizer, cfg: Config, steps_per_epoch: int):
    """
    Returns a LambdaLR that warms up linearly for `WARMUP_EPOCHS` epochs
    then follows a cosine annealing curve down to LR_MIN.
    """
    total_steps   = cfg.EPOCHS * steps_per_epoch
    warmup_steps  = cfg.WARMUP_EPOCHS * steps_per_epoch

    def lr_lambda(current_step: int) -> float:
        if current_step < warmup_steps:
            return float(current_step) / float(max(1, warmup_steps))
        progress = float(current_step - warmup_steps) / float(
            max(1, total_steps - warmup_steps)
        )
        cosine = 0.5 * (1.0 + math.cos(math.pi * progress))
        lr_ratio_min = cfg.LR_MIN / cfg.LR
        return max(lr_ratio_min, cosine)

    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

In [ ]:
# ── 7.3  Early stopping ───────────────────────────────────────────────────────

class EarlyStopping:
    def __init__(self, patience: int = CFG.PATIENCE, min_delta: float = CFG.MIN_DELTA, mode: str = "max"):
        self.patience   = patience
        self.min_delta  = min_delta
        self.mode       = mode
        self.best_score = None
        self.counter    = 0
        self.stop       = False

    def step(self, score: float) -> bool:
        if self.best_score is None:
            self.best_score = score
            return False

        if self.mode == "max":
            improved = score > self.best_score + self.min_delta
        else:
            improved = score < self.best_score - self.min_delta

        if improved:
            self.best_score = score
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.stop = True

        return self.stop

In [ ]:
# ── 7.4  One-epoch helpers ────────────────────────────────────────────────────

def train_one_epoch(
    model, loader, criterion, optimizer, scheduler, scaler, device, cfg
):
    model.train()
    total_loss = 0.0
    n_batches  = len(loader)

    for batch_idx, (specs, labels) in enumerate(loader):
        specs  = specs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with autocast():
            logits = model(specs)
            loss   = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), cfg.GRAD_CLIP)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        total_loss += loss.item()

    return total_loss / n_batches


@torch.no_grad()
def validate_one_epoch(model, loader, criterion, device):
    model.eval()
    total_loss  = 0.0
    all_logits  = []
    all_labels  = []

    for specs, labels in loader:
        specs  = specs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        with autocast():
            logits = model(specs)
            loss   = criterion(logits, labels)

        total_loss += loss.item()
        all_logits.append(logits.cpu().float())
        all_labels.append(labels.cpu().float())

    all_logits = torch.cat(all_logits).numpy()   # (N, C)
    all_labels = torch.cat(all_labels).numpy()   # (N, C)
    probs = 1 / (1 + np.exp(-all_logits))        # sigmoid

    # Macro ROC-AUC (only for classes present in the val set)
    try:
        auc = roc_auc_score(all_labels, probs, average="macro")
    except ValueError:
        auc = 0.0

    return total_loss / len(loader), auc, probs, all_labels

In [ ]:
# ── 7.5  Full training loop ───────────────────────────────────────────────────

def train(model, train_loader, val_loader, cfg: Config = CFG):
    criterion = BCEWithLabelSmoothing(smoothing=cfg.LABEL_SMOOTHING).to(DEVICE)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=cfg.LR, weight_decay=cfg.WEIGHT_DECAY
    )
    scheduler = build_scheduler(optimizer, cfg, len(train_loader))
    scaler    = GradScaler()
    es        = EarlyStopping(patience=cfg.PATIENCE, mode="max")

    best_auc  = 0.0
    ckpt_path = str(cfg.OUTPUT_DIR / f"best_model_fold{cfg.FOLD}.pt")
    history   = defaultdict(list)

    for epoch in range(1, cfg.EPOCHS + 1):
        t0 = time.time()

        train_loss = train_one_epoch(
            model, train_loader, criterion, optimizer, scheduler, scaler,
            DEVICE, cfg,
        )
        val_loss, val_auc, val_probs, val_labels = validate_one_epoch(
            model, val_loader, criterion, DEVICE
        )

        elapsed = time.time() - t0
        lr_now  = optimizer.param_groups[0]["lr"]

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_auc"].append(val_auc)
        history["lr"].append(lr_now)

        print(
            f"Epoch [{epoch:02d}/{cfg.EPOCHS}]  "
            f"train_loss: {train_loss:.4f}  "
            f"val_loss: {val_loss:.4f}  "
            f"val_AUC: {val_auc:.4f}  "
            f"lr: {lr_now:.2e}  "
            f"[{elapsed:.0f}s]"
        )

        if val_auc > best_auc:
            best_auc = val_auc
            torch.save({"model_state": model.state_dict(), "epoch": epoch, "auc": val_auc}, ckpt_path)
            print(f"  => Saved best model (AUC={best_auc:.4f}) at {ckpt_path}")

        if es.step(val_auc):
            print(f"  => Early stopping triggered after {epoch} epochs.")
            break

    print(f"\nTraining complete. Best val AUC: {best_auc:.4f}")
    return history, best_auc


# ── Run training ──────────────────────────────────────────────────────────────
history, best_auc = train(model, train_loader, val_loader, CFG)

In [ ]:
# ── 7.6  Training curves ──────────────────────────────────────────────────────

epochs_run = list(range(1, len(history["train_loss"]) + 1))

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

axes[0].plot(epochs_run, history["train_loss"], label="train")
axes[0].plot(epochs_run, history["val_loss"],   label="val")
axes[0].set_title("Loss")
axes[0].set_xlabel("Epoch")
axes[0].legend()

axes[1].plot(epochs_run, history["val_auc"], color="green")
axes[1].axhline(best_auc, linestyle="--", color="darkgreen", label=f"Best: {best_auc:.4f}")
axes[1].set_title("Validation Macro ROC-AUC")
axes[1].set_xlabel("Epoch")
axes[1].legend()

axes[2].plot(epochs_run, history["lr"], color="purple")
axes[2].set_title("Learning Rate Schedule")
axes[2].set_xlabel("Epoch")
axes[2].set_yscale("log")

plt.suptitle(f"Training History — Fold {CFG.FOLD}", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

---
## 8. Evaluation

Load the best checkpoint and compute comprehensive validation metrics:
- Macro ROC-AUC (competition metric)
- Micro & macro F1
- Per-class ROC-AUC (top-20 worst classes)
- Confusion matrix (top-20 most common species)

In [ ]:
# ── 8.1  Load best checkpoint ─────────────────────────────────────────────────
ckpt_path = str(CFG.OUTPUT_DIR / f"best_model_fold{CFG.FOLD}.pt")
ckpt = torch.load(ckpt_path, map_location=DEVICE)
model.load_state_dict(ckpt["model_state"])
model.eval()
print(f"Loaded checkpoint from epoch {ckpt['epoch']} | AUC={ckpt['auc']:.4f}")

In [ ]:
# ── 8.2  Full validation pass ─────────────────────────────────────────────────
criterion_eval = BCEWithLabelSmoothing(smoothing=0.0)  # No smoothing for eval
_, val_auc, val_probs, val_labels = validate_one_epoch(
    model, val_loader, criterion_eval, DEVICE
)

val_preds_binary = (val_probs >= CFG.SCORE_THRESHOLD).astype(int)

f1_micro = f1_score(val_labels, val_preds_binary, average="micro", zero_division=0)
f1_macro = f1_score(val_labels, val_preds_binary, average="macro", zero_division=0)

print("=" * 45)
print(f"  Validation Macro ROC-AUC : {val_auc:.4f}")
print(f"  Validation Micro F1      : {f1_micro:.4f}")
print(f"  Validation Macro F1      : {f1_macro:.4f}")
print("=" * 45)

In [ ]:
# ── 8.3  Per-class ROC-AUC — worst 20 species ─────────────────────────────────
per_class_auc = {}
for i, sp in enumerate(all_species):
    if val_labels[:, i].sum() > 0:  # skip classes absent in val set
        try:
            per_class_auc[sp] = roc_auc_score(val_labels[:, i], val_probs[:, i])
        except ValueError:
            per_class_auc[sp] = np.nan

auc_series = pd.Series(per_class_auc).sort_values(ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Bottom-20 (worst performing classes)
worst20 = auc_series.head(20)
axes[0].barh(worst20.index, worst20.values, color="tomato", edgecolor="white", lw=0.5)
axes[0].axvline(0.5, linestyle="--", color="black", linewidth=1)
axes[0].set_xlim(0, 1)
axes[0].set_title("20 Worst Species — Per-class ROC-AUC", fontsize=11)
axes[0].set_xlabel("ROC-AUC")

# Top-20 (best performing classes)
best20 = auc_series.tail(20).sort_values(ascending=False)
axes[1].barh(best20.index, best20.values, color="mediumseagreen", edgecolor="white", lw=0.5)
axes[1].set_xlim(0, 1)
axes[1].set_title("20 Best Species — Per-class ROC-AUC", fontsize=11)
axes[1].set_xlabel("ROC-AUC")

plt.suptitle("Per-class ROC-AUC Distribution", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

print(f"Median per-class AUC: {auc_series.median():.4f}")
print(f"Classes below 0.6 AUC: {(auc_series < 0.6).sum()} / {len(auc_series)}")

In [ ]:
# ── 8.4  Confusion matrix (top-20 species by val support) ────────────────────
# For multi-label we take argmax as the "primary prediction" for visualisation

top20_species = (
    val_df["primary_label"].value_counts().head(20).index.tolist()
)
top20_idx = [species_to_idx[sp] for sp in top20_species if sp in species_to_idx]

# Ground-truth primary label index
gt_primary = np.argmax(val_labels[:, top20_idx], axis=1)
pred_primary = np.argmax(val_probs[:, top20_idx], axis=1)

cm = confusion_matrix(gt_primary, pred_primary, labels=list(range(len(top20_idx))))
cm_norm = cm.astype(float) / (cm.sum(axis=1, keepdims=True) + 1e-9)

fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(
    cm_norm,
    annot=True, fmt=".2f",
    xticklabels=[sp[:12] for sp in top20_species],
    yticklabels=[sp[:12] for sp in top20_species],
    cmap="Blues",
    linewidths=0.5,
    ax=ax,
    vmin=0, vmax=1,
    annot_kws={"size": 7},
)
ax.set_title("Normalised Confusion Matrix — Top-20 Species (Val)", fontsize=13, fontweight="bold")
ax.set_xlabel("Predicted Label")
ax.set_ylabel("True Label")
plt.xticks(rotation=45, ha="right", fontsize=8)
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout()
plt.show()

---
## 9. Inference & Submission

Kaggle test soundscapes are long recordings that need to be chunked into 5-second windows at regular intervals. For each window, we predict probabilities and optionally apply **Test-Time Augmentation (TTA)** by averaging predictions over several augmented versions of the same clip.

In [ ]:
# ── 9.1  Chunk soundscape into 5-second windows ───────────────────────────────

def chunk_soundscape(
    path: str,
    sr: int = CFG.SR,
    duration: float = CFG.DURATION,
    step: float = 5.0,
):
    """
    Load a full soundscape and yield (start_sec, waveform_chunk) tuples.
    Each chunk is exactly `duration` seconds (padded if near end).
    `step` controls the stride between windows.
    """
    y, _ = librosa.load(path, sr=sr, mono=True, res_type="kaiser_fast")
    total_samples = len(y)
    chunk_samples = int(duration * sr)
    step_samples  = int(step * sr)

    for start in range(0, total_samples, step_samples):
        end = start + chunk_samples
        chunk = y[start:end]
        chunk = pad_or_truncate(chunk.astype(np.float32), chunk_samples)
        yield start // sr, chunk  # (start_sec, audio_array)

In [ ]:
# ── 9.2  Single-chunk prediction (with optional TTA) ─────────────────────────

IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
IMAGENET_STD  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)


def preprocess_chunk(y: np.ndarray, cfg: Config = CFG) -> torch.Tensor:
    mel    = audio_to_melspec(y, cfg)
    spec   = melspec_to_tensor(mel, cfg.IMG_HEIGHT, cfg.IMG_WIDTH)
    spec   = (spec - IMAGENET_MEAN) / IMAGENET_STD
    return spec.unsqueeze(0)   # (1, 3, H, W)


@torch.no_grad()
def predict_chunk(
    model: nn.Module,
    y: np.ndarray,
    cfg: Config = CFG,
    tta_steps: int = 0,
) -> np.ndarray:
    """
    Return sigmoid probabilities for one audio chunk.
    TTA: average predictions over `tta_steps` time-shifted + noise versions.
    """
    model.eval()
    all_probs = []

    # Original
    tensor = preprocess_chunk(y, cfg).to(DEVICE)
    with autocast():
        logits = model(tensor)
    all_probs.append(torch.sigmoid(logits).cpu().float().numpy())

    # TTA augmented passes
    for _ in range(tta_steps):
        y_aug = time_shift(y)
        if random.random() < 0.5:
            y_aug = add_gaussian_noise(y_aug)
        tensor = preprocess_chunk(y_aug, cfg).to(DEVICE)
        with autocast():
            logits = model(tensor)
        all_probs.append(torch.sigmoid(logits).cpu().float().numpy())

    return np.mean(all_probs, axis=0).squeeze(0)  # (NUM_CLASSES,)

In [ ]:
# ── 9.3  Full inference over test soundscapes ────────────────────────────────

sample_sub = pd.read_csv(CFG.SAMPLE_SUB_CSV)
print(f"Sample submission shape: {sample_sub.shape}")
print(sample_sub.head())

In [ ]:
# Identify all test soundscape files
test_files = sorted(CFG.TEST_DIR.glob("*.ogg")) + sorted(CFG.TEST_DIR.glob("*.wav"))
print(f"Test soundscapes found: {len(test_files)}")

# Load best model weights
ckpt = torch.load(str(CFG.OUTPUT_DIR / f"best_model_fold{CFG.FOLD}.pt"), map_location=DEVICE)
model.load_state_dict(ckpt["model_state"])
model.eval()

predictions = []  # list of dicts: {row_id, sp1, sp2, ...}

for audio_path in test_files:
    soundscape_id = audio_path.stem
    print(f"  Processing: {soundscape_id}", end="\r")

    for start_sec, chunk in chunk_soundscape(str(audio_path), sr=CFG.SR, duration=CFG.DURATION):
        probs = predict_chunk(model, chunk, CFG, tta_steps=CFG.TTA_STEPS)
        row_id = f"{soundscape_id}_{start_sec + int(CFG.DURATION)}"

        row = {"row_id": row_id}
        for i, sp in enumerate(all_species):
            row[sp] = float(probs[i])
        predictions.append(row)

print(f"\nTotal prediction rows: {len(predictions)}")

In [ ]:
# ── 9.4  Build submission dataframe ──────────────────────────────────────────

sub_df = pd.DataFrame(predictions)

# Align to sample_submission column order
expected_cols = sample_sub.columns.tolist()
# Fill any missing species columns with 0
for col in expected_cols:
    if col not in sub_df.columns:
        sub_df[col] = 0.0
sub_df = sub_df[expected_cols]

print(f"Submission shape: {sub_df.shape}")
print(sub_df.head())

# Save
sub_path = str(CFG.OUTPUT_DIR / "submission.csv")
sub_df.to_csv(sub_path, index=False)
print(f"Saved submission to: {sub_path}")

In [ ]:
# ── 9.5  Submission distribution sanity check ────────────────────────────────

species_cols = [c for c in sub_df.columns if c != "row_id"]
avg_species_probs = sub_df[species_cols].mean(axis=0)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Distribution of mean probability across species
axes[0].hist(avg_species_probs.values, bins=50, color="steelblue", edgecolor="white", lw=0.5)
axes[0].set_title("Distribution of Mean Predicted Probability per Species")
axes[0].set_xlabel("Mean predicted probability")
axes[0].set_ylabel("Number of species")

# Top-20 species by mean predicted probability
top20_pred = avg_species_probs.sort_values(ascending=False).head(20)
axes[1].barh(top20_pred.index[::-1], top20_pred.values[::-1], color="coral", edgecolor="white", lw=0.4)
axes[1].set_title("Top-20 Species by Mean Predicted Probability")
axes[1].set_xlabel("Mean predicted probability")

plt.suptitle("Submission Sanity Check", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

print(f"Overall mean prediction probability: {sub_df[species_cols].values.mean():.4f}")
print(f"Any NaN in submission: {sub_df.isnull().any().any()}")

---
## Summary

| Step | Details |
|---|---|
| **Data** | `train_metadata.csv` + `.ogg` recordings; 234 species from Pantanal |
| **Audio** | 32 kHz mono, 5-second clips (repeat-pad / random crop) |
| **Features** | Mel spectrogram — 128 mel bins, N_FFT=2048, hop=512 |
| **Augmentation** | Time shift, Gaussian noise, mixup (waveform); SpecAugment (spectrogram) |
| **Backbone** | EfficientNet-B2 (timm, ImageNet pretrained) |
| **Head** | GeM pooling → Dropout(0.3) → Linear(234) |
| **Loss** | BCEWithLogitsLoss + label smoothing (0.05) |
| **Optimiser** | AdamW (lr=1e-3, wd=1e-4) |
| **Schedule** | Linear warmup (3 epochs) + cosine annealing |
| **Mixed precision** | `torch.cuda.amp` AMP |
| **Validation** | StratifiedKFold (5 folds) |
| **Metric** | Macro ROC-AUC |
| **Inference** | 5-second sliding window + TTA (5 steps) |

### Next steps

- Train all 5 folds and ensemble predictions with a weighted average.
- Experiment with larger backbones: `efficientnet_b4`, `tf_efficientnetv2_s`.
- Add background soundscape noise augmentation with real field recordings.
- Tune the GeM `p` parameter and dropout rate via optuna.
- Explore focal loss to address class imbalance.